In [4]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np

# Parameters
input_dim = 20
latent_dim = 2

# ---------------- ENCODER ----------------
encoder_inputs = layers.Input(shape=(input_dim,))
x = layers.Dense(10, activation='relu')(encoder_inputs)

z_mean = layers.Dense(latent_dim)(x)
z_log_var = layers.Dense(latent_dim)(x)

class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = Sampling()([z_mean, z_log_var])

encoder = Model(encoder_inputs, [z_mean, z_log_var, z])

# ---------------- DECODER ----------------
latent_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(10, activation='relu')(latent_inputs)
decoder_outputs = layers.Dense(input_dim, activation='sigmoid')(x)

decoder = Model(latent_inputs, decoder_outputs)

# ---------------- VAE MODEL ----------------
class VAE(Model):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):
        if isinstance(data, tuple):
            data = data[0]

        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)

            # Reconstruction loss
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(tf.square(data - reconstruction), axis=1)
            )

            # KL loss
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                    axis=1
                )
            )

            total_loss = reconstruction_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        return {
            "loss": total_loss,
            "reconstruction_loss": reconstruction_loss,
            "kl_loss": kl_loss,
        }

# Build & compile
vae = VAE(encoder, decoder)
vae.compile(optimizer='adam')

# Dummy data
x_train = np.random.rand(1000, input_dim).astype("float32")

# Train
vae.fit(x_train, epochs=20, batch_size=32)

# Generate samples
z_sample = np.random.normal(size=(10, latent_dim)).astype("float32")
generated = decoder.predict(z_sample)

print("Generated Data:\n", generated)

Epoch 1/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - kl_loss: 0.5300 - loss: 2.3864 - reconstruction_loss: 1.8563
Epoch 2/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - kl_loss: 0.0950 - loss: 1.9016 - reconstruction_loss: 1.8067
Epoch 3/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - kl_loss: 0.0567 - loss: 1.6072 - reconstruction_loss: 1.5506
Epoch 4/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - kl_loss: 0.0816 - loss: 1.6967 - reconstruction_loss: 1.6151
Epoch 5/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - kl_loss: 0.0841 - loss: 1.9374 - reconstruction_loss: 1.8533
Epoch 6/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - kl_loss: 0.0531 - loss: 1.7781 - reconstruction_loss: 1.7250
Epoch 7/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - kl_loss: 0.0497 - loss: 1.6374 - reconstruction_loss: 1.5877
Epoch 8/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - kl_loss: 0.0403 - loss: 1.7460 - reconstruction_loss: 1.7057 
Epoch 9/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - kl_loss: 0.0298 - loss: 1.8293 - reconstruc

In [5]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np

# ---------------- GENERATOR ----------------
def build_generator():
    model = tf.keras.Sequential([
        layers.Dense(16, activation='relu', input_shape=(10,)),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

# ---------------- DISCRIMINATOR ----------------
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Dense(16, activation='relu', input_shape=(1,)),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

generator = build_generator()
discriminator = build_discriminator()

# Compile discriminator
discriminator.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Freeze discriminator for GAN
discriminator.trainable = False

# GAN Model
gan_input = layers.Input(shape=(10,))
fake = generator(gan_input)
gan_output = discriminator(fake)

gan = Model(gan_input, gan_output)
gan.compile(optimizer='adam', loss='binary_crossentropy')

# ---------------- TRAINING ----------------
epochs = 1000
batch_size = 32

for epoch in range(epochs):

    # Real data
    real_data = np.random.rand(batch_size, 1)
    real_labels = np.ones((batch_size, 1))

    # Fake data
    noise = np.random.rand(batch_size, 10)
    fake_data = generator.predict(noise, verbose=0)
    fake_labels = np.zeros((batch_size, 1))

    # Train Discriminator
    discriminator.trainable = True
    d_loss_real = discriminator.train_on_batch(real_data, real_labels)
    d_loss_fake = discriminator.train_on_batch(fake_data, fake_labels)

    # Train Generator
    discriminator.trainable = False
    noise = np.random.rand(batch_size, 10)
    g_loss = gan.train_on_batch(noise, np.ones((batch_size, 1)))

    if epoch % 200 == 0:
        print(f"Epoch {epoch}, D Loss: {d_loss_real[0]+d_loss_fake[0]}, G Loss: {g_loss}")

# Generate samples
noise = np.random.rand(5, 10)
generated = generator.predict(noise)
print("Generated Data:\n", generated)

c:\Users\prate\Desktop\ZTMmine\env\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 0, D Loss: 1.7937042713165283, G Loss: 1.0073959827423096
Epoch 200, D Loss: 1.449819803237915, G Loss: 0.7546150088310242
Epoch 400, D Loss: 1.405559778213501, G Loss: 0.6928682923316956
Epoch 600, D Loss: 1.4046118259429932, G Loss: 0.6775255799293518
Epoch 800, D Loss: 1.3979612588882446, G Loss: 0.7083514928817749
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Generated Data:
 [[0.4472649 ]
 [0.37521863]
 [0.39932904]
 [0.3811341 ]
 [0.36539602]]


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

# ---------------- MODEL ----------------
class GCN(nn.Module):
    def __init__(self):
        super(GCN, self).__init__()
        self.fc1 = nn.Linear(10, 5)
        self.fc2 = nn.Linear(5, 2)

    def forward(self, x, adj):
        x = torch.matmul(adj, x)     # Message passing
        x = torch.relu(self.fc1(x))
        x = torch.matmul(adj, x)
        x = self.fc2(x)
        return x

# ---------------- DATA ----------------
num_nodes = 6
features = torch.rand(num_nodes, 10)

# Adjacency matrix (with self-loops)
adj = torch.eye(num_nodes)

# Labels (2 classes)
labels = torch.randint(0, 2, (num_nodes,))

# ---------------- TRAINING ----------------
model = GCN()
optimizer = optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(50):
    optimizer.zero_grad()

    output = model(features, adj)
    loss = loss_fn(output, labels)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

# ---------------- PREDICTION ----------------
pred = torch.argmax(model(features, adj), dim=1)
print("Predicted Classes:", pred)

Epoch 0, Loss: 0.7815338969230652
Epoch 10, Loss: 0.7213395237922668
Epoch 20, Loss: 0.6870740056037903
Epoch 30, Loss: 0.6637303233146667
Epoch 40, Loss: 0.6495599150657654
Predicted Classes: tensor([1, 1, 1, 1, 1, 1])
